# Impulse + Consolidation

The classic "impulse + pause" micro-pattern.

One directional "impulse" candle (large body + above-average volume, closing in the top/bottom third of its range) \
followed by 2-4 smaller "consolidation" candles whose aggregate range sits inside a fraction of the impulse range \
and holds above/below the 50% retrace of the impulse.

__Pattern definition__

Impulse candle (C₀):
- Body ≥ 1.5 × the average body of the prior 20 candles
- Volume ≥ 1.2 × volume SMA(9)
- Closes in the upper third (bullish) or lower third (bearish) of its range

Consolidation (C₁ … Cₙ, where n = 2–4):
- Each candle's body ≤ 50% of C₀'s body
- The entire cluster's high–low range ≤ 70% of C₀'s range
- Cluster holds above the 50% retrace of C₀ (for bull flags) / below for bear flags
- No single candle closes beyond C₀'s opposite extreme

Pattern expiry:
- If no breakout by Cₙ₊₃, the setup is void.

__How Impulse + Consolidation Algorithm Determines Entry/Exit:__

Track A - Continuation WITH higher-timeframe trend (trade WITH the impulse):
- Entry: stop order at the impulse extreme (break-out side): stop-buy at C₀ high + 1 tick (long) / stop-sell at C₀ low − 1 tick (short).
- Skipped if price is within 0.3% of the 24H high/low (sweep risk).
- Stop: below the cluster low (or above cluster high for shorts), with buffer = max(0.5 × ATR(14), swing wick).
- Target: T1 = 1R (scale half), T2 = measured move of C₀ projected from breakout.

Only valid when all three context filters agree:
- Price is on the correct side of EMA20 and EMA20 slope matches the impulse direction
- Higher timeframe (1H) structure is HH/HL for longs or LH/LL for shorts
- Not within 0.3% of a major S/R or round number (avoids sweep zones)

Track B - Failure fade AGAINST a counter-trend impulse (trade AGAINST the impulse):
- Entry: stop order at the far side of the consolidation (the break of the cluster, not the impulse).
- Stop: above C₀ high (or below C₀ low).
- Target: previous swing / EMA20 pullback / opposite side of the recent range

Only valid when context filters disagree with the impulse (i.e. impulse is counter-trend):
- Impulse is a green candle but 1H is LH/LL and price is under EMA20, OR
- Impulse happens at 24H high/low / obvious liquidity level (likely sweep).

__Hard filters (no-trade conditions):__
- Volume on C₀ < volume SMA → skip. The impulse is fake.
- Consolidation forms through EMA20 rather than against it → skip (structure is breaking).
- R:R to the first structural target < 1.5 → skip.
- Funding rate extreme in the direction of the impulse (crowd is already long/short) → skip Track A, upgrade Track B.
- Within 15 minutes of a scheduled macro event → skip.

On a 5m chart in a clean trending session these patterns print every 30–60 minutes. \
Most will not pass the filter stack. That's the point — the algorithm's edge comes from throwing most of them away. \
If the filter is passing more than ~30% of detected patterns, the filter is too loose.

__Risk:__
- ATR-scaled stop buffer (max of 0.5 * ATR(14) or the cluster wick).
- Fixed fractional risk per trade (default 0.25% of equity).
- Min 1.5 : 1 reward-to-risk gate on the averaged target.
- T1 at 1R (scale 50% + move to break-even), T2 at max(2R, measured move).
- Breakout entries expire if unfilled within 3 bars.
- One trade per cluster. If stopped, do not re-enter the same pattern.

__Further analysis:__
1. Label the population. \
Pull 3–6 months of 5m BTC data, detect every instance matching the pattern definition. Expect a few thousand.
2. Separate by context filter. \
Bucket into "with-trend" and "counter-trend" groups.
3. Compute expectancy per bucket.\
The raw pattern is a coin-flip (~50%, negative after fees). \
With-trend + volume confirmation should push win rate to 55–60% with avg R around 1.3–1.6. \
Counter-trend fades should show similar.
4. Check correlation of losers. \
If most losses cluster around specific hours (e.g. Asia open chop) or at specific levels (round numbers), add a time/level filter.
5. Walk-forward on the last month only before live-testing with minimum size.

## Configuration

In [1]:
# --- path bootstrap: make 'engine' importable from any CWD ---
import sys, pathlib
root = pathlib.Path.cwd()
while not (root / "engine" / "__init__.py").exists():
    if root == root.parent: raise RuntimeError("project root not found")
    root = root.parent
if str(root) not in sys.path: sys.path.insert(0, str(root))

In [2]:
from engine.backtester import Backtester
from engine.trade_configurator import ACTIVE_TRADE
from engine.core import Signal, SignalAction
from engine.strategy_configurator import StrategyConfig
from engine.visualization import build_chart

In [ ]:
# Dataset is configured globally in engine/data_configurator.py (the ACTIVE spec).
# Edit ACTIVE there to change symbol / interval / window for every notebook at once.
from engine.data_configurator import ACTIVE, save_result

SYMBOL, INTERVAL = ACTIVE.symbol, ACTIVE.interval

In [ ]:
# Candles come from the shared cache via the global ACTIVE spec.
from engine.data_configurator import load_data

df = load_data()

## Impulse + Consolidation

In [5]:
# Import Impulse strategy
from engine.strategies import ImpulseFlagStrategy


In [ ]:
# Backtest Impulse strategy
config = StrategyConfig()
strategy = ImpulseFlagStrategy(config)
bt = Backtester(strategy, symbol=SYMBOL, trading_config=ACTIVE_TRADE)
result = bt.run(df, interval=INTERVAL)
print(result.summary())

# Save metrics (JSON) + trade log (CSV) under data/results/<dataset signature>/
save_result(result, ACTIVE)

In [ ]:
# Dollar P&L
INITIAL_BALANCE = 100  # USD

balance = INITIAL_BALANCE
peak = balance
max_dd = 0

print(f"\n{'─' * 45}")
print(f"  Dollar P&L (starting ${INITIAL_BALANCE:,.2f})")
print(f"{'─' * 45}")

for i, t in enumerate(result.trades, 1):
    prev = balance
    balance *= (1 + t.pnl_bps / 10_000)
    peak = max(peak, balance)
    max_dd = max(max_dd, (peak - balance) / peak)
    pnl = balance - prev
    print(f"  #{i:3d}  {t.direction.value:5s}  {pnl:+8.2f}  →  ${balance:,.2f}")

print(f"{'─' * 45}")
print(f"  Final balance  : ${balance:,.2f}")
print(f"  Net profit     : ${balance - INITIAL_BALANCE:+,.2f}")
print(f"  Return         : {(balance / INITIAL_BALANCE - 1):+.2%}")
print(f"  Max drawdown   : {max_dd:.2%}")

In [ ]:
# Impulse strategy chart
prepared = strategy.prepare(df)
signals = []
for t in result.trades:
    if t.entry_ts:
        signals.append(Signal(t.entry_ts, SignalAction.ENTRY, t.direction, t.entry_price))
    if t.exit_ts:
        signals.append(Signal(t.exit_ts, SignalAction.EXIT, t.direction, t.exit_price))

build_chart(
    prepared, signals,
    title=f"{SYMBOL} {INTERVAL}m | {strategy.name} | {result.total_trades} trades",
).show()

## Live signals

Live mode:
- It generates signals, it does not place orders. \
There's no exchange API key, no order execution. It tells you when to enter/exit.
- State persists \
If you stop and restart, it remembers whether you're in a position via its SQLite state file under data/live/.
- Circuit breaker \
If Bybit is unreachable 10 times in a row, it stops automatically instead of spinning forever.
- Chart updates in place \
Automatic: the chart refreshes in the browser every poll_seconds.

To actually execute trades automatically, you need to add authenticated Bybit order placement on top of the signal output.

If you don't want to re-download automatically, two edits locally:
1. visualization.py — add auto_refresh: int = 0 parameter to build_chart, and after fig.write_html(save_path):
pythonif auto_refresh > 0:
    with open(save_path, "r") as f:
        html = f.read()
    meta_tag = f'<meta http-equiv="refresh" content="{auto_refresh}">'
    html = html.replace("<head>", f"<head>{meta_tag}", 1)
    with open(save_path, "w") as f:
        f.write(html)
2. live.py — add auto_refresh=self.poll_seconds to the build_chart() call in _tick().

### From CLI (command line interface)

- runs in a loop
- polls (refreshes) Bybit every 30 seconds
- persists state to SQLite (survives restarts)
- writes a chart under data/live/ each tick
- handles SIGTERM/Ctrl+C gracefully

In [ ]:
python -m engine \
    --strategy impulse_flag \
    --mode live \
    --symbol BTCUSDT \
    --interval 15 \
    --candles 500 \
    --poll 30

### From a notebook cell

In [ ]:
from engine.live import LiveEngine
from engine.data_configurator import LIVE_DIR
from engine.strategy_configurator import StrategyConfig
from engine.strategies import ImpulseFlagStrategy

SYMBOL   = "BTCUSDT"
INTERVAL = "15"

config = StrategyConfig()
strategy = ImpulseFlagStrategy(config)

engine = LiveEngine(
    strategy=strategy,
    symbol=SYMBOL,
    interval=INTERVAL,
    num_candles=500,
    poll_seconds=30,
    chart_path=str(LIVE_DIR / f"{SYMBOL}_{INTERVAL}_{strategy.name}.html"),
    db_path=str(LIVE_DIR / f"{strategy.name}.db"),
)

engine.run()  # blocks until Ctrl+C or kernel interrupt